In [5]:
import pandas as pd
import glob

In [7]:
# Data Quality Validation
files = glob.glob("data/raw/games/games_*.csv")

results = []

# For each data file of games, we will check various data points for the file
for file in files:

    games = pd.read_csv(file)

    fbs_games = games[
        (games["homeClassification"] == "fbs") | (games["awayClassification"] == "fbs")
    ].copy()

    results.append({
        "season": games["season"].iloc[0],
        "all_games": len(games),
        "fbs_games": len(fbs_games),
        "fbs_vs_fbs": (
            (fbs_games["homeClassification"] == "fbs") & (fbs_games["awayClassification"] == "fbs")
        ).sum(),
        "fbs_vs_fcs": (
            (
                (fbs_games["homeClassification"] == "fbs") & (fbs_games["awayClassification"] == "fcs")
            ) |
            (
                (fbs_games["homeClassification"] == "fcs") & (fbs_games["awayClassification"] == "fbs")
            )
        ).sum(),
        "missing_home_class": games["homeClassification"].isna().sum(),
        "missing_away_class": games["awayClassification"].isna().sum(),
        "duplicate_ids": games["id"].duplicated().sum(),
        "missing_home_score": games["homePoints"].isna().sum(),
        "missing_away_score": games["awayPoints"].isna().sum()
    })

# Report the results
audit = pd.DataFrame(results).sort_values("season")

print(audit.to_string(index = False))

 season  all_games  fbs_games  fbs_vs_fbs  fbs_vs_fcs  missing_home_class  missing_away_class  duplicate_ids  missing_home_score  missing_away_score
   2015       1491        829         724         105                   0                  12              0                   0                   0
   2016       1502        832         719         113                   1                  11              0                   0                   0
   2017       1505        834         736          98                   1                  10              0                   0                   0
   2018       1511        845         733         112                   1                  14              0                   0                   0
   2019       1577        848         734         114                   0                  23              0                   0                   0
   2020        563        542         508          34                   0                   0             

In [9]:
# The quality check showed 8 games with missing scores for one or both teams. Investigate those games
files = glob.glob("data/raw/games/games_*.csv")

for file in files:

    games = pd.read_csv(file)

    missing_scores = games[
        games["homePoints"].isna() |
        games["awayPoints"].isna()
    ]

    if len(missing_scores) > 0:

        print(f"\n{'=' * 60}")
        print(f"SEASON: {games['season'].iloc[0]}")
        print(f"{'=' * 60}")

        print(
            missing_scores[
                [
                    "id",
                    "season",
                    "week",
                    "seasonType",
                    "completed",
                    "startDate",
                    "homeTeam",
                    "homeClassification",
                    "awayTeam",
                    "awayClassification",
                    "homePoints",
                    "awayPoints",
                    "notes"
                ]
            ].to_string(index=False)
        )


SEASON: 2023
       id  season  week seasonType  completed                startDate        homeTeam homeClassification         awayTeam awayClassification  homePoints  awayPoints notes
401552878    2023     9    regular      False 2023-10-28T17:00:00.000Z           Colby                iii       Middlebury                iii         NaN         NaN   NaN
401550299    2023     9    regular      False 2023-10-28T17:00:00.000Z           Bates                iii         Williams                iii         NaN         NaN   NaN
401549719    2023     9    regular      False 2023-10-28T17:00:00.000Z         Bowdoin                iii     Trinity (CT)                iii         NaN         NaN   NaN
401552884    2023    11    regular      False 2023-11-12T17:00:00.000Z Worcester State                iii Framingham State                iii         NaN         NaN   NaN

SEASON: 2024
       id  season  week seasonType  completed                startDate  homeTeam homeClassification         away

In [ ]:
# After this data check, the master data file for games will follow the following 3 guidlines:
# 1. Keep only games involving one or more FBS teams
# 2. Keep only completed games
# 3. Require a final score for both the home and away team in order to pass through to the final dataset

In [13]:
# Moving on to stats data, checking information about the source prior to cleaning
df = pd.read_csv("data/raw/stats/team_stats_2025.csv")

print(df.shape)
print(df.head())

print("\nUnique teams:", df["team"].nunique())
print("\nUnique statistics:", df["statName"].nunique())

print("\nStatistics:")
print(df["statName"].unique())

print("\nMissing values:")
print(df.isna().sum())

print("\nRecords per team:")
print(df.groupby("team").size().describe())

(8568, 5)
   season       team     conference                       statName  statValue
0    2025  Air Force  Mountain West                     firstDowns        264
1    2025  Air Force  Mountain West             firstDownsOpponent        247
2    2025  Air Force  Mountain West          fourthDownConversions         24
3    2025  Air Force  Mountain West  fourthDownConversionsOpponent         10
4    2025  Air Force  Mountain West                    fourthDowns         34

Unique teams: 136

Unique statistics: 63

Statistics:
<StringArray>
[                   'firstDowns',            'firstDownsOpponent',
         'fourthDownConversions', 'fourthDownConversionsOpponent',
                   'fourthDowns',           'fourthDownsOpponent',
                   'fumblesLost',           'fumblesLostOpponent',
              'fumblesRecovered',      'fumblesRecoveredOpponent',
                         'games',                 'interceptions',
         'interceptionsOpponent',               'in